# Projecte Final: Anàlisi del finançament de les administracions públiques de Catalunya als mitjans de comunicació (2015-2024)

## Notebook 2 ETL-Extracció de dades — Menjòmetre (Premsa, 2015-2025)

**Com funciona:** La pàgina és Next.js. Les dades estan dins del bundleJavaScript amb cometes escapades com `\"`. Un sol `replace` les converteix
en JSON parsejable, i `json.loads()` extreu tots els camps de forma robusta.

Captura **totes** les entitats (individuals i de grup) amb el nom net. Cada entitat del JSON del Menjòmetre porta un camp `by_admin` que desglossa el seu import TOTAL per administració finançadora (Generalitat, diputacions, ajuntaments...), el captura i l'afegeix com a columnes noves.


In [1]:
import pandas as pd


### 1. **Transformació del dataset** amb 4 columnes noves:
   - `Nom_comercial`: com es coneix realment el mitjà (ex. "El Punt Avui"
     en lloc de "HERMES COMUNICACIONS,S.A.")
   - `Tipus_mitja`: Premsa / Ràdio / TV / Internet / Agència de notícies /
     Associació-Entitat / Productora / Altres
   - `Ambit`: Catalunya / Espanya / Local-Comarcal / Comunitat Valenciana /
     Illes Balears
   - `Govern_Generalitat`: partit(s) que governaven aquell any 


   1. Diccionari de categorització — `Nom_comercial`, `Tipus_mitja`, `Ambit`

La clau del mapeig és el **CIF** (cada CIF té un únic nom associat al dataset, verificat prèviament). Les entitats no cobertes pel diccionari
manual es categoritzen amb l'heurística de fallback.
                                                                            

In [2]:
## carreguem el DF
df = pd.read_csv('menjometre_detallat_admin.csv', encoding='utf-8-sig')

print(f'Files: {df.shape[0]} | Columnes: {df.columns.tolist()}')
print(f"Anys: {sorted(df['Any'].unique())}")


Files: 974 | Columnes: ['Any', 'Mitja', 'CIF', 'Grup', 'Subvencions', 'Contractes', 'Pub. institucional', 'Total', 'admin_ajuntament_bcn', 'admin_altres', 'admin_altres_ajuntaments', 'admin_diputacio_bcn', 'admin_diputacio_girona', 'admin_diputacio_lleida', 'admin_diputacio_tarragona', 'admin_estat', 'admin_generalitat']
Anys: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [3]:
# -*- coding: utf-8 -*-
"""
Diccionari de categorització de les entitats del dataset del Menjòmetre.
Clau: CIF (fiable, cada CIF té un únic nom associat al dataset).
Valor: (nom_comercial, tipus_mitja, ambit)
"""

CATEGORITZACIO_PER_CIF = {
    # --- Grans grups i mitjans de referència ---
    "A08849622": ("TV3 (CCMA)",                         "TV",       "Catalunya"),
    "B61475257": ("La Vanguardia",                       "Premsa",   "Catalunya"),
    "B66485343": ("El Periódico de Catalunya",            "Premsa",   "Catalunya"),
    "B65258261": ("Ara",                                  "Premsa",   "Catalunya"),
    "B61726626": ("RAC1",                                 "Ràdio",    "Catalunya"),
    "A17374547": ("El Punt Avui",                         "Premsa",   "Catalunya"),
    "A17374647": ("El Punt Avui",                         "Premsa",   "Catalunya"), ##Possible error
    "B61801635": ("Flaix (Flaixbac / Flaix FM)",          "Ràdio",    "Catalunya"),
    "B28016970": ("Cadena SER",                           "Ràdio",    "Espanya"),
    "B66541947": ("El Nacional",                           "Internet", "Catalunya"),
    "A63200828": ("8TV",                                    "TV",       "Catalunya"),
    "F64074982": ("Sàpiens",                              "Premsa",   "Catalunya"),
    "A08846974": ("Catalunya Ràdio",                      "Ràdio",    "Catalunya"),
    "A08775487": ("Catalunya Ràdio",                      "Ràdio",    "Catalunya"),
    "B60955564": ("Nació Digital",                        "Internet", "Catalunya"),
    "B25275926": ("Segre",                                "Premsa",   "Local/Comarcal"),
    "A08605503": ("Sport",                                "Premsa",   "Catalunya"),
    "A28782936": ("Onda Cero / Europa FM",                "Ràdio",    "Espanya"),
    "G61514162": ("AMIC (Associació de Mitjans)",         "Associació/Entitat", "Catalunya"),
    "B65672495": ("Premsa comarcal (80 Mes 4 Publicacions)", "Premsa", "Local/Comarcal"),
    "B99083966": ("20 Minutos",                           "Premsa",   "Espanya"),
    "A61135521": ("Diari de Girona",                      "Premsa",   "Local/Comarcal"),
    "G58366675": ("APPEC (Associació editors premsa)",    "Associació/Entitat", "Catalunya"),
    "G58335158": ("APPEC (Associació editors premsa)",    "Associació/Entitat", "Catalunya"),
    "A43056787": ("El Temps",                             "Premsa",   "Comunitat Valenciana"),
    "A08539165": ("Regió 7",                               "Premsa",   "Local/Comarcal"),
    "A59114082": ("Mundo Deportivo",                       "Premsa",   "Espanya"),
    "A08025009": ("Mundo Deportivo",                       "Premsa",   "Espanya"),
    "A08447369": ("El 9 Nou",                               "Premsa",   "Local/Comarcal"),
    "B08936643": ("Godó Strategies (serveis Grup Godó)",   "Altres",   "Catalunya"),
    "B85635910": ("El País",                               "Premsa",   "Espanya"),
    "B61017810": ("VilaWeb",                               "Internet", "Catalunya"),
    "A28281368": ("Cadena COPE",                           "Ràdio",    "Espanya"),
    "A75000190": ("Cadena COPE",                           "Ràdio",    "Espanya"),
    "B02851848": ("Diari de Terrassa",                     "Premsa",   "Local/Comarcal"),
    "A08911745": ("Radio Tele Taxi",                       "Ràdio",    "Local/Comarcal"),
    "A08536922": ("Europa Press Catalunya",                "Agència de notícies", "Catalunya"),
    "A28078343": ("Europa Press",                          "Agència de notícies", "Espanya"),
    "A28122076": ("Condé Nast (Vogue, GQ...)",             "Premsa",   "Espanya"),
    "A28028744": ("Agència EFE",                           "Agència de notícies", "Espanya"),
    "A86436195": ("Antena 3 (Atresmedia)",                 "TV",       "Espanya"),
    "A79075438": ("Telecinco / Cuatro (Mediaset)",         "TV",       "Espanya"),
    "B86509254": ("Premsa digital (Diario de Prensa Digital)", "Internet", "Espanya"),
    "B64610389": ("RBA Revistas",                          "Premsa",   "Espanya"),
    "B66195207": ("RBA Llibres",                           "Premsa",   "Espanya"),
    "G46648655": ("Fundació Francesc Eiximenis",           "Associació/Entitat", "Catalunya"),
    "B67186866": ("Premsa/digital local (Novapress)",      "Internet", "Local/Comarcal"),
    "B64863335": ("Enderrock",                             "Premsa",   "Catalunya"),
    "B09844770": ("Premsa local digital (grup)",           "Internet", "Local/Comarcal"),   ##Possiblement grup Nació Digital
    "B58364936": ("Totmedia Comunicació (Grup Hermes)",    "Premsa",   "Catalunya"),
    "A28297059": ("Grupo PRISA",                           "Premsa",   "Espanya"),
    "A88096458": ("Grupo PRISA",                           "Premsa",   "Espanya"),
    "G58010315": ("Associació Catalana de Premsa Comarcal","Associació/Entitat", "Catalunya"),
    "A46186821": ("Premsa Comunitat Valenciana (Edicions)","Premsa",   "Comunitat Valenciana"),
    "A43926682": ("Tamediaxa",                             "Premsa",   "Local/Comarcal"),
    "A60783686": ("Barcelona TV (ICB)",                    "TV",       "Local/Comarcal"),
    "R5800622B": ("Fundació Blanquerna (URL)",             "Associació/Entitat", "Catalunya"),
    "B66566415": ("Crónica Global",                        "Internet", "Catalunya"),
    "B81230764": ("Hearst España (revistes)",              "Premsa",   "Espanya"),
    "B43121144": ("El Vallenc",                            "Premsa",   "Local/Comarcal"),
    "G04980595": ("Associació Espai Línia",                "Associació/Entitat", "Catalunya"),
    "B25503566": ("Edicions Saloria",                      "Premsa",   "Local/Comarcal"),
    "B17091711": ("El Empordà",                            "Premsa",   "Local/Comarcal"),
    "B65331704": ("Comunicació Vinaròs",                   "Premsa",   "Comunitat Valenciana"),
    "A25004474": ("La Mañana (Lleida)",                    "Premsa",   "Local/Comarcal"),
    "F65947814": ("Cultura 21",                            "Associació/Entitat", "Catalunya"),
    "B64337462": ("El Singular Digital",                   "Internet", "Catalunya"),
    "G78982733": ("AIMC",                                  "Associació/Entitat", "Espanya"),
    "B08503583": ("L'Avenç",                               "Premsa",   "Catalunya"),
    "F66334806": ("Crític",                                "Internet", "Catalunya"),
    "B65078545": ("Capgròs (Empordà)",                     "Internet", "Local/Comarcal"),
    "G65978348": ("Fundació Periodisme Plural",            "Associació/Entitat", "Catalunya"),
    "B64166705": ("Racó Català",                           "Internet", "Catalunya"),
    "B43808294": ("Limícola",                              "Altres",   "Local/Comarcal"),
    "B82399327": ("Multiprensa (premsa esportiva)",        "Premsa",   "Espanya"),
    "B55053482": ("Directe.cat",                           "Internet", "Catalunya"),
    "B66917048": ("Núvol",                                 "Internet", "Catalunya"),
    "B67273219": ("Quelcom Global",                        "Altres",   "Catalunya"),
    "B57447849": ("Prensa Ibérica (digital)",              "Internet", "Espanya"),
    "A08605503_dup": None,  # placeholder no usat

    # --- Prensa Ibérica (grup) ---
    "A61135521_dup": None,

    # --- CCMA relacionats ---
    "A08849622_dup": None,

    # --- Altres agències / distribuïdores ---
    "B86150332": ("El Mundo / Unidad Editorial",           "Premsa",   "Espanya"),
    "B85076900": ("Unidad Editorial (informació econ.)",   "Premsa",   "Espanya"),
    "B82824194": ("ABC",                                   "Premsa",   "Espanya"),
    "B85088052": ("Radio Marca",                           "Ràdio",    "Espanya"),
    "A82031329": ("esRadio",                               "Ràdio",    "Espanya"),
    "A28602389": ("Motorpress Ibérica (revistes)",         "Premsa",   "Espanya"),
    "A07136472": ("Premsa Illes Balears (Prensa Ibicenca)","Premsa",   "Illes Balears"),
    "A07016732": ("Premsa Illes Balears (Editora Balear)", "Premsa",   "Illes Balears"),
    "B66236373": ("Time Out",                              "Premsa",   "Espanya"),
    "A08176499": ("Publicacions de l'Abadia de Montserrat","Premsa",   "Catalunya"),
    "A43030302": ("Diari de Tarragona (Grup)",             "Premsa",   "Local/Comarcal"),
}


def _netejar_nom_fallback(nom: str) -> str:
    """Neteja bàsica del nom original per fer-lo servir com a nom comercial
    quan no hi ha una entrada explícita al diccionari."""
    net = nom.strip()
    # Treure formes jurídiques habituals per llegibilitat
    for sufix in [", S.L.U.", ", SLU", ", S.L.", ", SL", ", S.A.", ", SA",
                  " S.L.U.", " SLU", " S.L.", " SL", " S.A.", " SA",
                  ", SCCL", " SCCL", ", SCP"]:
        if net.upper().endswith(sufix.upper()):
            net = net[: -len(sufix)].strip()
            break
    return net.title()


def categoritzar_heuristic(nom: str):
    """
    Heurística de fallback per a entitats NO cobertes pel diccionari manual
    (majoritàriament associacions culturals locals, mitjans comarcals
    petits i entitats amb un import molt reduït).
    """
    n = nom.upper()

    paraules_associacio = ["ASSOCIACIO", "ASSOC.", "ASS.", "FUNDACIO", "FUND.",
                            "FEDERACIO", "SINDICAT", "COL.LEGI", "COL.",
                            "SOCIETAT", "INSTITUT"]
    paraules_radio = ["RADIO", "RÀDIO", " FM", "FM ", "FM,", "FM."]
    paraules_tv = [" TV", "TELEVISIO", "TELEVISIÓ", "AUDIOVISUAL"]
    paraules_internet = ["DIGITAL", ".CAT", "WEB", "ONLINE", ".COM", "XARXA"]
    paraules_productora = ["PRODUCCIONS", "PRODUCCIO"]
    paraules_altres = ["PUBLICITARI", "CONSULTANTS", "STRATEGY", "REPUTATION"]

    if any(p in n for p in paraules_associacio):
        tipus = "Associació/Entitat"
    elif any(p in n for p in paraules_radio):
        tipus = "Ràdio"
    elif any(p in n for p in paraules_tv):
        tipus = "TV"
    elif any(p in n for p in paraules_internet):
        tipus = "Internet"
    elif any(p in n for p in paraules_productora):
        tipus = "Productora"
    elif any(p in n for p in paraules_altres):
        tipus = "Altres"
    else:
        tipus = "Premsa"  # categoria per defecte per a la resta (majoria premsa local)

    ambit = "Local/Comarcal"  # per defecte, ja que la majoria son entitats petites/locals
    nom_comercial = _netejar_nom_fallback(nom)

    return nom_comercial, tipus, ambit


def categoritzar_entitat(cif: str, nom: str):
    """
    Retorna (nom_comercial, tipus_mitja, ambit) per a una entitat.
    Prioritza el diccionari manual per CIF; si no hi és, aplica l'heurística.
    """
    if cif and cif in CATEGORITZACIO_PER_CIF and CATEGORITZACIO_PER_CIF[cif] is not None:
        return CATEGORITZACIO_PER_CIF[cif]
    return categoritzar_heuristic(nom)


### 2 **Enriquim dataset** Diccionari del partit governant — `Govern_Generalitat`

In [4]:
# -*- coding: utf-8 -*-
"""
Partit(s) que van governar la Generalitat de Catalunya per any.
Com que les dades del Menjòmetre són anuals (no mensuals), els anys de
transició de govern es marquen explícitament amb els dos partits i el mes
de canvi, en lloc d'assignar-los arbitràriament a un sol partit.
"""

GOVERN_PER_ANY = {
    2015: "CiU",
    2016: "'Junts pel Sí'",
    2017: "Junts pel Sí/155",
    2018: "155/JxCat",
    2019: "JxCat",
    2020: "JxCat/funcions",
    2021: "funcions/ERC",
    2022: "ERC",
    2023: "ERC",
    2024: "ERC/PSC"
}



### 3. Apliquem la categorització i transformem el dataset enriquit

In [5]:
## Executem funcions de mapeig fila per fila. Resultats retornar els valors de la clau del diccionari el CIF, que és una tupla amb "nom", "tipologia" i "ambit"
resultats = df.apply(
    lambda r: categoritzar_entitat(r['CIF'] if pd.notna(r['CIF']) else '', r['Mitja']), ## Omplim
    axis=1
)
df['Nom_comercial'] = [r[0] for r in resultats]   ##Creem la columna
df['Tipus_mitja']   = [r[1] for r in resultats]
df['Ambit']          = [r[2] for r in resultats]
df['Govern_Generalitat'] = df['Any'].map(GOVERN_PER_ANY)


display(df.head(10))


,Any,Mitja,CIF,Grup,Subvencions,Contractes,Pub. institucional,Total,admin_ajuntament_bcn,admin_altres,...,admin_diputacio_bcn,admin_diputacio_girona,admin_diputacio_lleida,admin_diputacio_tarragona,admin_estat,admin_generalitat,Nom_comercial,Tipus_mitja,Ambit,Govern_Generalitat
0,2015,"EL PERIODICO DE CATALUNYA, SLU",B66485343,Prensa Ibérica,0.0,0.0,3659692.01,3659692.01,0.0,231181.90,...,0.0,0.0,0.0,0.0,0.0,3428510.11,El Periódico de Catalunya,Premsa,Catalunya,CiU
1,2015,"HERMES COMUNICACIONS,S.A.",A17374547,Hermes Comunicacions (Punt Avui),0.0,0.0,3552221.00,3552221.00,0.0,147594.83,...,0.0,0.0,0.0,0.0,0.0,3404626.17,El Punt Avui,Premsa,Catalunya,CiU
2,2015,"CORPORACIO CATALANA DE MITJANS AUDIOVISUALS, S.A.",A08849622,CCMA (Mitjans públics catalans),0.0,0.0,2406820.03,2406820.03,0.0,1534209.45,...,0.0,0.0,0.0,0.0,0.0,872610.58,TV3 (CCMA),TV,Catalunya,CiU
3,2015,"EDICIO DE PREMSA PERIODICA ARA, SL",B65258261,Diari ARA,0.0,0.0,2387072.21,2387072.21,0.0,68344.35,...,0.0,0.0,0.0,0.0,0.0,2318727.86,Ara,Premsa,Catalunya,CiU
4,2015,"LA VANGUARDIA EDICIONES,S.L.",B61475257,Grup Godó,0.0,0.0,1920371.65,1920371.65,0.0,242057.18,...,0.0,0.0,0.0,0.0,0.0,1678314.47,La Vanguardia,Premsa,Catalunya,CiU
5,2015,"RADIOCAT XXI,S.L.",B61726626,Grup Godó,0.0,0.0,1612474.13,1612474.13,0.0,501274.44,...,0.0,0.0,0.0,0.0,0.0,1111199.69,RAC1,Ràdio,Catalunya,CiU
6,2015,"EMISSIONS DIGITALS DE CATALUNYA, SA",A63200828,NaN,0.0,0.0,1476817.60,1476817.60,0.0,290538.10,...,0.0,0.0,0.0,0.0,0.0,1186279.50,8TV,TV,Catalunya,CiU
7,2015,"GRUP FLAIX, SL",B61801635,NaN,0.0,0.0,516456.15,516456.15,0.0,115244.27,...,0.0,0.0,0.0,0.0,0.0,401211.88,Flaix (Flaixbac / Flaix FM),Ràdio,Catalunya,CiU
8,2015,Atresmedia Corporación de Medios de Comunicaci...,A86436195,NaN,0.0,0.0,367707.87,367707.87,0.0,239518.57,...,0.0,0.0,0.0,0.0,0.0,128189.30,Antena 3 (Atresmedia),TV,Espanya,CiU
9,2015,"MEDIASET ESPAÑA COMUNICACIÓN, S.A.",A79075438,NaN,0.0,0.0,336479.82,336479.82,0.0,238736.94,...,0.0,0.0,0.0,0.0,0.0,97742.88,Telecinco / Cuatro (Mediaset),TV,Espanya,CiU


In [6]:
##Verifiquem que s'ha transformat el DataFrame correctament
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Any                        974 non-null    int64  
 1   Mitja                      974 non-null    str    
 2   CIF                        961 non-null    str    
 3   Grup                       138 non-null    str    
 4   Subvencions                974 non-null    float64
 5   Contractes                 974 non-null    float64
 6   Pub. institucional         974 non-null    float64
 7   Total                      974 non-null    float64
 8   admin_ajuntament_bcn       974 non-null    float64
 9   admin_altres               974 non-null    float64
 10  admin_altres_ajuntaments   974 non-null    float64
 11  admin_diputacio_bcn        974 non-null    float64
 12  admin_diputacio_girona     974 non-null    float64
 13  admin_diputacio_lleida     974 non-null    float64
 14  admin

### 4. Modifiquem "Grup" d'algun dels mitjans

S'assignen a grups diferents mitjans que en l'extracció del Menjometre no tenien cap grup assignat, però que realment pertanyen als quals s'assignarà. A la resta mitjans sense Grup Assignat, se'ls hi assigna el mateix Nom_comercial del mitjà per no tenir NAN, i facilitin l'anàlisi estadístic

In [7]:

afegir_a_grups = {
    "A08775487": "CCMA (Mitjans públics catalans)",
    "A59114082": "Grup Godó",  
    "A08025009": "Grup Godó",
    "B60955564": "Nació Digital",
    "B09844770": "Nació Digital",
    "B85635910": "Grupo Prisa",
    "A88096458": "Grupo Prisa",
    "B28016970": "Grupo Prisa",
    "A28297059": "Grupo Prisa"
}

df["Grup"] = df["CIF"].map(afegir_a_grups).fillna(df["Grup"])

In [8]:
df["Grup"] = df["Grup"].fillna(df["Nom_comercial"])


### 5. Exportem el Dataset

In [9]:
df.to_csv('menjometre_detallat_enriquit_admin.csv', index=False, encoding='utf-8-sig')
print('Guardat: menjometre_detallat_enriquit_admin.csv')

Guardat: menjometre_detallat_enriquit_admin.csv
